# AI in Medicine and Healthcare - Class 8
## Signal Basics: Convolutions and Image Smoothing

**Prof. Dr. Marcel P. Jackowski**

**Insper Institute of Education and Research**

---

### Learning Objectives

By the end of this lab, you will be able to:

1. Understand signal noise in medical images
2. Apply convolution operations for image smoothing
3. Compare different smoothing kernels
4. Evaluate the Signal-to-Noise Ratio (SNR)

---

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q numpy matplotlib scipy scikit-image pillow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, signal
from skimage import data
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['image.cmap'] = 'gray'

## 2. Understanding Signals and Noise

### 2.1 Theory Review

Medical images are **2D signals** f(x,y) where:
- **Signals** are mathematical functions representing physical processes
- **Noise** corrupts the signal: y = x + e
- **SNR** (Signal-to-Noise Ratio) measures signal quality

Medical imaging modalities introduce different types of noise:
- **Gaussian noise**: thermal noise in electronic systems
- **Salt-and-pepper noise**: dead pixels
- **Poisson noise**: quantum noise in X-ray/CT imaging

### 2.2 Load Sample Image

In [ ]:
#
# Load a multiframe DICOM image fro previous labs and select one slice as your sample image!
#
# If you download a DICOM dataset of your own, make sure you share the link it with the instructor!
#
print("Loading medical imaging data...")
## TO DO:
## medical_image = ...
##
print(f"Image shape: {medical_image.shape}")
print(f"Value range: [{medical_image.min():.1f}, {medical_image.max():.1f}]")

In [ ]:
# Display the clean image
plt.figure(figsize=(10, 10))
plt.imshow(medical_image, cmap='gray')
plt.title('Original Medical Image (No Noise)', fontsize=14, fontweight='bold')
plt.axis('off')
plt.colorbar(label='Intensity')
plt.tight_layout()
plt.show()

### 2.3 Adding Different Types of Noise

In [ ]:
def add_gaussian_noise(image, sigma=25):
    """Add Gaussian noise to simulate thermal noise"""
    noise = np.random.normal(0, sigma, image.shape)
    noisy_image = image + noise
    return np.clip(noisy_image, 0, 255)

def add_salt_pepper_noise(image, amount=0.05):
    """Add salt-and-pepper noise to simulate dead pixels"""
    noisy_image = image.copy()
    # Salt
    num_salt = int(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy_image[coords[0], coords[1]] = 255
    # Pepper
    num_pepper = int(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy_image[coords[0], coords[1]] = 0
    return noisy_image

def add_poisson_noise(image):
    """Add Poisson noise to simulate quantum noise in X-ray/CT"""
    normalized = image / 255.0
    noisy = np.random.poisson(normalized * 100) / 100.0
    return np.clip(noisy * 255, 0, 255)

# Create noisy versions
noisy_gaussian = add_gaussian_noise(medical_image, sigma=30)
noisy_sp = add_salt_pepper_noise(medical_image, amount=0.02)
noisy_poisson = add_poisson_noise(medical_image)

# Display all noise types
fig, axes = plt.subplots(2, 2, figsize=(15, 15))

axes[0, 0].imshow(medical_image, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original (Clean)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_gaussian, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Gaussian Noise', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(noisy_sp, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Salt-and-Pepper Noise', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(noisy_poisson, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('Poisson Noise', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

### 2.4 Signal Profile Analysis

In [ ]:
# Extract a horizontal line from the middle of the image
row_idx = medical_image.shape[0] // 2
clean_profile = medical_image[row_idx, :]
noisy_profile = noisy_gaussian[row_idx, :]

# Plot profiles
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

axes[0].imshow(noisy_gaussian, cmap='gray')
axes[0].axhline(y=row_idx, color='red', linewidth=2, label='Profile Line')
axes[0].set_title('Noisy Image with Profile Line', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].axis('off')

axes[1].plot(clean_profile, linewidth=2, color='blue')
axes[1].set_title('Clean Signal Profile P(x)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Position (x)')
axes[1].set_ylabel('Intensity')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 300])

axes[2].plot(noisy_profile, linewidth=1, color='red', alpha=0.7, label='Noisy')
axes[2].plot(clean_profile, linewidth=2, color='blue', alpha=0.5, label='Clean', linestyle='--')
axes[2].set_title('Noisy Signal Profile', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Position (x)')
axes[2].set_ylabel('Intensity')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 300])

plt.tight_layout()
plt.show()

## 3. Discrete Convolution for Smoothing

### 3.1 Understanding Convolution

**Discrete convolution** is the sum of products with each possible shift of a kernel. A **kernel** (also called a **filter**) is a small matrix of weights used in convolution.

For **smoothing**, we can use averaging kernels that reduce noise.

For simple averaging, we want a kernel where:
- Every element has the same value
- All elements sum to 1 (to preserve brightness)

**For a 3×3 averaging kernel:**
```
[[1/9, 1/9, 1/9],
 [1/9, 1/9, 1/9],
 [1/9, 1/9, 1/9]]
```

**Your task:** Write a function that creates a `size × size` averaging kernel.

**Requirements:**
- Each element should be `1 / (size²)`
- Do **not** use loops — research NumPy array constructors
- Return a NumPy array

### 3.2 Define Smoothing Kernels

In [ ]:
# Box filter kernels (uniform averaging)
#
# TO DO:
# Replace the lines below to create each kernel size
#
kernel_3x3 = np.zeros((3,3))
kernel_5x5 = np.zeros((5,5))
kernel_7x7 = np.zeros((7,7))
#

#
# This is a more complex kernel!
#
# Gaussian kernel
#
def gaussian_kernel(size=5, sigma=1.0):
    kernel = np.fromfunction(
        lambda x, y: (1/(2*np.pi*sigma**2)) * np.exp(-((x-(size-1)/2)**2 + (y-(size-1)/2)**2)/(2*sigma**2)),
        (size, size)
    )
    return kernel / kernel.sum()

kernel_gaussian = gaussian_kernel(5, 1.0)

# Display kernels
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

im1 = axes[0].imshow(kernel_3x3, cmap='hot', interpolation='nearest')
axes[0].set_title('Box 3x3', fontsize=11, fontweight='bold')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(kernel_5x5, cmap='hot', interpolation='nearest')
axes[1].set_title('Box 5x5', fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(kernel_7x7, cmap='hot', interpolation='nearest')
axes[2].set_title('Box 7x7', fontsize=11, fontweight='bold')
plt.colorbar(im3, ax=axes[2])

im4 = axes[3].imshow(kernel_gaussian, cmap='hot', interpolation='nearest')
axes[3].set_title('Gaussian 5x5', fontsize=11, fontweight='bold')
plt.colorbar(im4, ax=axes[3])

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Box 3x3 kernel:")
print(kernel_3x3)

print("Box 5x5 kernel:")
print(kernel_5x5)

print("Box 3x3 kernel:")
print(kernel_3x3)

### Convolution Function

We'll use SciPy's `correlate2d` instead of `convolve2d` because some kernels are **not symmetric**, and we don't want them to be flipped.

**Why `correlate2d`?**
- Convolution flips the kernel before applying it
- Correlation does not flip the kernel
- For symmetric kernels (like Gaussian), they're the same
- For asymmetric kernels (like edge detection kernels), we want correlation

### 3.3 Apply Convolution

In [ ]:
# Apply different smoothing kernels
smoothed_3x3 = signal.correlate2d(noisy_gaussian, kernel_3x3, mode='valid')
smoothed_5x5 = signal.correlate2d(noisy_gaussian, kernel_5x5, mode='valid')
smoothed_7x7 = signal.correlate2d(noisy_gaussian, kernel_7x7, mode='valid')
smoothed_gaussian = signal.correlate2d(noisy_gaussian, kernel_gaussian, mode='valid')

# Display results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(medical_image, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original (Clean)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_gaussian, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Noisy', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(smoothed_3x3, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title('Smoothed (Box 3x3)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].imshow(smoothed_5x5, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Smoothed (Box 5x5)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(smoothed_7x7, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('Smoothed (Box 7x7)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(smoothed_gaussian, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title('Smoothed (Gaussian 5x5)', fontsize=12, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

What does the `mode='valid'` parameter do ?

### 3.4 Compare Profiles

In [ ]:
# Extract profiles
profile_3x3 = smoothed_3x3[row_idx, :]
profile_5x5 = smoothed_5x5[row_idx, :]
profile_gaussian = smoothed_gaussian[row_idx, :]

# Plot comparison
plt.figure(figsize=(15, 6))
plt.plot(clean_profile, linewidth=3, color='blue', label='Original Clean', alpha=0.7)
plt.plot(noisy_profile, linewidth=1, color='red', label='Noisy', alpha=0.5)
plt.plot(profile_3x3, linewidth=2, color='green', label='Smoothed 3x3', linestyle='--')
plt.plot(profile_5x5, linewidth=2, color='orange', label='Smoothed 5x5', linestyle='--')
plt.plot(profile_gaussian, linewidth=2, color='purple', label='Smoothed Gaussian', linestyle='-.')
plt.title('Signal Profile Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Position (x)', fontsize=12)
plt.ylabel('Intensity', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0, 300])
plt.tight_layout()
plt.show()

## 4. Signal-to-Noise Ratio (SNR)

In [ ]:
def calculate_snr(clean_signal, noisy_signal):
    """Calculate Signal-to-Noise Ratio"""
    noise = noisy_signal - clean_signal
    signal_rms = np.sqrt(np.mean(clean_signal**2))
    noise_rms = np.sqrt(np.mean(noise**2))
    snr = signal_rms / noise_rms if noise_rms > 0 else float('inf')
    snr_db = 20 * np.log10(snr) if snr > 0 else 0
    return snr, snr_db

def calculate_psnr(clean_signal, noisy_signal, max_value=255):
    """Calculate Peak Signal-to-Noise Ratio"""
    mse = np.mean((clean_signal - noisy_signal)**2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(max_value / np.sqrt(mse))
    return psnr

# Calculate SNR
results = {}
images_dict = {
    'Noisy': noisy_gaussian,
    'Smoothed 3x3': smoothed_3x3,
    'Smoothed 5x5': smoothed_5x5,
    'Smoothed 7x7': smoothed_7x7,
    'Smoothed Gaussian': smoothed_gaussian
}

for name, img in images_dict.items():
    snr, snr_db = calculate_snr(medical_image, img)
    psnr = calculate_psnr(medical_image, img)
    results[name] = {'SNR': snr, 'SNR_dB': snr_db, 'PSNR': psnr}

# Display results
print("\n" + "="*70)
print("Signal Quality Metrics")
print("="*70)
print(f"{'Image Type':<25} {'SNR':>10} {'SNR (dB)':>12} {'PSNR (dB)':>12}")
print("-"*70)

for name, metrics in results.items():
    print(f"{name:<25} {metrics['SNR']:>10.2f} {metrics['SNR_dB']:>12.2f} {metrics['PSNR']:>12.2f}")

print("="*70)

## 5. Practical Exercises

### Exercise 1: Optimal Kernel Size

In [ ]:
# Test different kernel sizes
kernel_sizes = [3, 5, 7, 9, 11, 13]
snr_results = []

for size in kernel_sizes:
    kernel = np.ones((size, size)) / (size * size)
    smoothed = signal.correlate2d(noisy_gaussian, kernel, mode='valid')
    _, snr_db = calculate_snr(medical_image, smoothed)
    snr_results.append(snr_db)

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(kernel_sizes, snr_results, marker='o', linewidth=2, markersize=8)
plt.xlabel('Kernel Size', fontsize=12)
plt.ylabel('SNR (dB)', fontsize=12)
plt.title('SNR vs. Kernel Size', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(kernel_sizes)

best_idx = np.argmax(snr_results)
best_size = kernel_sizes[best_idx]
plt.axvline(x=best_size, color='red', linestyle='--', label=f'Optimal: {best_size}x{best_size}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Optimal kernel size: {best_size}x{best_size}")
print(f"Maximum SNR: {snr_results[best_idx]:.2f} dB")

**Analysis**

In [ ]:
## 
## What was the optimal kernel size ? Can you explain why ? 
##

### Exercise 2: Noise Level Impact

In [ ]:
# Test different noise levels
noise_levels = [10, 20, 30, 40, 50]
snr_before = []
snr_after = []

for sigma in noise_levels:
    noisy = add_gaussian_noise(medical_image, sigma=sigma)
    _, snr_before_db = calculate_snr(medical_image, noisy)
    snr_before.append(snr_before_db)
    
    smoothed = signal.correlate2d(noisy, kernel_5x5, mode='valid')
    _, snr_after_db = calculate_snr(medical_image, smoothed)
    snr_after.append(snr_after_db)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(noise_levels, snr_before, marker='o', label='Before Smoothing', linewidth=2)
plt.plot(noise_levels, snr_after, marker='s', label='After Smoothing', linewidth=2)
plt.xlabel('Noise Level', fontsize=12)
plt.ylabel('SNR (dB)', fontsize=12)
plt.title('Noise Level Impact', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

improvements = [after - before for before, after in zip(snr_before, snr_after)]
print("SNR Improvement (dB):")
for sigma, improvement in zip(noise_levels, improvements):
    print(f"  Sigma={sigma}: +{improvement:.2f} dB")

In [ ]:
## 
## Test different kernels, including gaussian. How did they compare ?
##

## 6. Summary

### Key Takeaways:

1. **Signal Noise in Medical Imaging**
   - Gaussian, salt-and-pepper, and Poisson noise
   - Different sources and characteristics

2. **Convolution for Smoothing**
   - Box filters: uniform averaging
   - Gaussian filters: weighted averaging
   - Larger kernels = more smoothing = more blur

3. **Quality Metrics**
   - SNR measures signal quality
   - Trade-off: noise reduction vs detail preservation

### Clinical Relevance:

- Better SNR leads to clearer features and more accurate diagnosis
- Lower radiation dose produces more noise requiring better denoising
- Clean images improve AI/ML model performance